# Recovery of the true partition

Sections 5.3 and 5.4 of the paper: Figures 4, 5 and 6, and Figures 9 and 10 of the appendix.

Two experiments on the same mixtures and the same dissimilarity matrices: the Adjusted Rand
Index against the true partition $\mathcal P^\star$ along a path of increasing $n$ at fixed $N$,
then over the $(\alpha, K)$ grid of `assumptions.ipynb`. Section 5.4 drops the assumption that
$K$ is known and reads it off the dendrogram with the largest-gap rule (6).

**Design.** Mixtures are drawn as in `assumptions.ipynb`: $\Sigma = \{0,\dots,4\}$, first-order
chains with rows i.i.d. Dirichlet($\alpha$), $N$ sequences of length $n$ with i.i.d. latent
labels and balanced weights $w_k = 1/K$, each sequence started from its own uniformly drawn
Dirac law. The cost scheme is $c_{\mathrm{sub}} \equiv 2$, $\delta \equiv 1$, fixed in advance.
Each dissimilarity matrix $(\hat\gamma_n(i,j))_{i,j \le N}$ is clustered three times with $K$
known — single linkage, average linkage (SciPy), and $K$-medoids by PAM — so the three
estimators are compared on the same data and the differences are attributable to the algorithm
alone. Besides the mean ARI we report the exact-recovery event $\{\mathrm{ARI} = 1\}$, which is
the event Theorems 3.3 and 3.8 bound.

**Figures.** All are written to `Figures/Recovery/`.

| figure of the paper | file |
|---|---|
| 4a, 4b | `ari_path_average_linkage`, `ari_path_kmedoids` |
| 5a, 5b | `ari_grid_average_linkage`, `ari_grid_kmedoids` |
| 6 | `k_hat_grid` |
| 9, 10 (appendix, single linkage) | `ari_path_single_linkage`, `ari_grid_single_linkage` |

**Runtime.** One $N \times N$ matrix costs $\binom{N}{2}$ alignments of $O(n^2)$ each, and nested
prefixes share no work: the two-row dynamic program keeps no table to snapshot, so a matrix
evaluated over a grid of horizons costs the *sum* of the $n_g^2$ and not the largest of them. At
$N = 800$, $n = 1000$ a matrix takes about $45$ s on eight cores; the grid is $72$ cells $\times$
$30$ repetitions and accounts for nearly all of the $\approx 30$ h the notebook takes. Its
outputs are therefore not stored in the file — the figures above are.


In [ ]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from om_lib import (PAPER_STYLE, SEQUENTIAL_CMAP, adjusted_rand_index, average_linkage_labels,
                    check_assumption_metric, cost_scheme, cut_at_k, kmedoids, largest_gap_k,
                    om_matrices, sample_markov_model, sample_mixture, separation_levels,
                    single_linkage_tree, spectral_gap, stationary_distribution_markov)


## Setup

The first cell holds every parameter of the notebook — alphabet and costs, the $(N, n)$ path of
Section 5.3, the $(\alpha, K)$ grid of the same section — and the second the two evaluation
functions the experiments share: one reads a dissimilarity matrix, the other applies it along
nested prefixes of a sample.


In [ ]:
# --------------------------------------------------------------- shared ----
D_STATES     = 5           # alphabet size, Sigma = {0, ..., 4}
SEED         = 20260803
SUB_COST     = 2.0         # c_sub(a, b) for a != b, hence M = 2
INDEL_COST   = 1.0         # delta(a)
EXACT_TOL    = 1e-12       # ARI > 1 - EXACT_TOL is the exact-recovery event {ARI = 1}
MED_RESTARTS = 10          # random restarts of K-medoids, on top of PAM's BUILD
MED_SEED     = 0           # fixed, so the clustering is a function of D alone

# ------------------------------- experiment 1: the horizons, at fixed N ----
ALPHA_FIX = 0.3            # moderate randomness: distinguishable chains that still mix fast
K_FIX     = 4              # the largest K where separation still holds most of the time
R_PATH    = 50             # replicates, each running the whole set of horizons
PATH_N    = 800            # sequences, fixed throughout -- the same N as the grid below

#: The horizons, read on nested prefixes of the same PATH_N sequences. Only the length moves,
#: so a curve is one dataset growing in n and nothing can be lost from one point to the next
#: but sampling noise.
PATH_LEN  = np.arange(100, 1001, 100)

# ---------------------------------- experiment 2: the (alpha, K) grid ------
#: The same (alpha, K) grid as `assumptions.ipynb`, run at the same N as the path above and at
#: its last horizon, both tied to it here rather than transcribed, so that the grid sits on a
#: point the path actually visits.
ALPHAS  = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 5.0, 10.0])
KS      = [2, 3, 4, 5, 6, 7, 8, 9, 10]
N_SWEEP = PATH_N                                   # the same N as the path
GRID_N  = np.array([PATH_LEN[-1]])                 # the path's last n
R_SWEEP = 30                                       # repetitions per cell
LAST    = GRID_N.size - 1                          # the horizon the figures show

# --------------------------------------------------------------------------
S_COST, DELTA_COST = cost_scheme("constant", D_STATES, sub=SUB_COST, indel=INDEL_COST)
assert all(v for k, v in check_assumption_metric(S_COST, DELTA_COST).items()
           if k != "M = max c_sub")


In [ ]:
#: everything `evaluate_matrix` returns, in the order the sweeps store it
EVAL_KEYS = ("ari", "ari_average", "ari_medoids", "ari_k_hat", "k_hat",
             "in", "out", "out_max")


def evaluate_matrix(D, labels, K):
    """The three ARIs, the largest-gap estimate of K and the plug-in separation levels.

    Two partitions come from the same tree, so the second one is free -- the merge heights are
    already there:

    * `ari`, the cut at the number of non-empty classes, i.e. the number of blocks of P*. This
      is the estimator Theorem 3.3 is about, and it assumes K known.
    * `ari_k_hat`, the cut at `k_hat`, the largest-gap estimator (6) of K, governed by
      Assumption 4. This is what one can do without knowing K.

    Two more come from other algorithms on the same matrix: `ari_average`, average linkage, a
    bracketed linkage covered by Remark 3.4, and `ari_medoids`, K-medoids, the estimator of
    Theorem 3.8.

    Returns the three ARIs, k_hat, and Delta_in, Delta_out, Delta_out^max.
    """
    K_eff = int(np.unique(labels).size)
    heights, edges = single_linkage_tree(D)
    k_hat = largest_gap_k(heights)
    med_labels, _, _ = kmedoids(D, K_eff, np.random.default_rng(MED_SEED),
                                n_restarts=MED_RESTARTS)
    out = {"ari": adjusted_rand_index(cut_at_k(edges, D.shape[0], K_eff), labels),
           "ari_average": adjusted_rand_index(average_linkage_labels(D, K_eff), labels),
           "ari_medoids": adjusted_rand_index(med_labels, labels),
           "k_hat": float(k_hat),
           "ari_k_hat": adjusted_rand_index(cut_at_k(edges, D.shape[0], k_hat), labels)}
    out.update(separation_levels(D, labels, K))
    return out


def evaluate_paths(X, labels, grid, K):
    """`evaluate_matrix` along nested prefixes of X: arrays indexed by the horizon grid."""
    Ds = om_matrices(X, grid, S_COST, DELTA_COST)
    out = {key: np.empty(len(grid)) for key in EVAL_KEYS}
    for g in range(len(grid)):
        ev = evaluate_matrix(Ds[g], labels, K)
        for key in out:
            out[key][g] = ev[key]
    return out

## The ARI along nested prefixes, at fixed $N$

Section 5.3, Figures 4a, 4b and 9. The exact-recovery bound of the paper,
$N^2\exp(-\varepsilon^2 n/(2C^\star))$, decays exponentially in $n$ at fixed $N$, which is the
statement this figure illustrates: $N = 800$ throughout — the same $N$ as the grid below — and
only the horizon moves.

Ten horizons, $n = 100, 200, \dots, 1000$, are read on **nested prefixes of the same $N$
sequences**: a replicate draws its $800$ sequences once, at full length, and every point of the
curve reads them at a shorter prefix. A curve is therefore a genuine sample path of one dataset
growing in length, and not a sequence of independent draws; nothing can be lost from one point
to the next but sampling noise, so a drop is a property of the data and not an artefact of the
design. $R = 50$ replicates, at $K = 4$ and $\alpha = 0.3$.


In [ ]:
rng = np.random.default_rng(SEED)
KERNELS = np.stack([sample_markov_model(D_STATES, 1, ALPHA_FIX, rng)["transitions"]
                    for _ in range(K_FIX)])

gaps = np.array([spectral_gap(P) for P in KERNELS])
print("spectral gaps    :", np.round(gaps, 3))
print("relaxation times :", np.round(1.0 / gaps, 1))
print("stationary laws  :\n", np.round(np.stack([stationary_distribution_markov(P)
                                                 for P in KERNELS]), 3))
print(f"\nhorizons : {PATH_LEN.tolist()}   at the fixed N = {PATH_N}")


In [ ]:
def sweep_path(N, ns, kernels, alpha, K, R, seed=SEED + 1):
    """The three ARIs, k_hat and the separation levels along nested prefixes, at fixed N.

    One replicate draws N sequences of length ns[-1] once and reads them at every horizon, so a
    curve is a sample path of one dataset growing in length. A replicate costs sum_g n_g^2 and
    not the largest horizon: `om_matrices` re-runs the alignment at every length.
    """
    rng = np.random.default_rng(seed)
    out = {key: np.empty((R, ns.size)) for key in EVAL_KEYS}
    t0 = time.time()
    for r in tqdm(range(R), desc="replicate"):
        mix = sample_mixture(K, N, int(ns[-1]), D_STATES, alpha, rng, kernels=kernels)
        ev = evaluate_paths(mix["X"], mix["labels"], ns, K)
        for key in out:
            out[key][r] = ev[key]
    print(f"{R} replicates over {ns.size} horizons in {time.time() - t0:.0f}s")
    return {"N": N, "n": ns, **out}


path = sweep_path(PATH_N, PATH_LEN, KERNELS, ALPHA_FIX, K_FIX, R_PATH)

In [ ]:
def crossing(x, y, level=0.5):
    """Abscissa at which y first reaches `level`, by linear interpolation."""
    y = np.asarray(y, dtype=float)
    idx = int(np.argmax(y >= level))
    if y[idx] < level or idx == 0:
        return np.nan
    x0, x1, y0, y1 = x[idx - 1], x[idx], y[idx - 1], y[idx]
    return float(x0 + (level - y0) * (x1 - x0) / (y1 - y0))


mean_ari = path["ari"].mean(0)
margin = path["out"] - path["in"]

print(f"N = {path['N']} sequences at every horizon, {path['ari'].shape[0]} replicates\n")
ESTIMATORS = (("ari", "single"), ("ari_average", "average"), ("ari_medoids", "medoids"))

print(f"{'n':>7}" + "".join(f"{name:>10}" for _, name in ESTIMATORS)
      + "   |" + "".join(f"{'P(=1) ' + name:>14}" for _, name in ESTIMATORS)
      + f"{'margin':>10}")
for t in range(path["n"].size):
    means = "".join(f"{path[key][:, t].mean():>10.3f}" for key, _ in ESTIMATORS)
    exact = "".join(f"{(path[key][:, t] > 1 - EXACT_TOL).mean():>14.2f}" for key, _ in ESTIMATORS)
    print(f"{path['n'][t]:>7}{means}   |{exact}{np.nanmedian(margin[:, t]):>+10.3f}")

print()
for key, name in ESTIMATORS:
    m = path[key].mean(0)
    bits = []
    for lvl in (0.5, 0.99):
        if m[0] >= lvl:
            bits.append(f"{lvl:.2f} already at n = {path['n'][0]}")
        else:
            c = crossing(path["n"], m, lvl)
            bits.append(f"{lvl:.2f} at n = {c:.0f}" if not np.isnan(c)
                        else f"{lvl:.2f} not reached")
    print(f"{name:>8}: mean ARI " + ", ".join(bits))

### The three estimators, on the same matrices

Every dissimilarity matrix is read by the three clustering algorithms, at every horizon and in
every repetition, so the comparisons are on the same data. It costs nothing: the matrix is what
is expensive, and each clustering takes a few hundredths of a second next to the tens of seconds
it costs to build one.

| estimator | cut / criterion | what covers it |
|---|---|---|
| single linkage | first $N-K$ merges of the MST | Theorem 3.3 |
| average linkage | SciPy, cut at $K$ blocks | Remark 3.4, as a bracketed linkage |
| $K$-medoids | minimiser of $\Phi_{N,n}$, by PAM | Theorem 3.8 |

Average linkage is the one used in practice, for stability reasons, and it also isolates *why*
single linkage fails when it fails: single linkage merges two blocks on the **smallest**
dissimilarity between them, so one aberrant sequence chains two true classes together, and the
cut at $K$ then spends a whole block on that sequence — the ARI collapses to a fixed level
($0.712$ for $K = 4$ balanced) instead of degrading smoothly. Average linkage merges on the mean
over all pairs, where no single sequence carries the decision.


In [ ]:
COL = {"ari": "#1b6ca8",           # the blue of om_convergence.ipynb
       # a third hue, so that the three estimators never share a colour
       "ari_average": "#6a51a3",
       "ari_medoids": "#c0392b"}
COL_PATH, COL_GAP = COL["ari"], COL["ari_medoids"]


def plot_ari_curve(path, key, name, filename=None):
    """Sample paths of the ARI of one estimator against the horizon, at fixed N.

    Same layout as `plot_gamma_convergence` in om_convergence.ipynb: one thin line per
    replicate, the mean over them on top. A replicate is a sample path -- the same N sequences
    read at growing lengths -- so the spread between the thin lines is the variability of a
    single experiment, not an interval around the mean. The three estimators share these axes
    exactly, so the figures can be stacked and compared.
    """
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        ax.plot(path["n"], path[key].T, color=COL[key], lw=0.6, alpha=0.16)
        ax.plot(path["n"], path[key].mean(0), color=COL[key], lw=1.9,
                label=rf"mean over {path[key].shape[0]} replicates")
        ax.set_xlabel(r"$n$, length of a sequence")
        ax.set_ylabel(f"ARI, {name}")
        ax.set_ylim(-0.04, 1.04)
        ax.axhline(0.5, color="0.6", lw=0.7, ls=(0, (1, 3)), zorder=0)
        ax.set_title(rf"{name} --- $N = {path['N']}$, $K = {K_FIX}$, $\alpha = {ALPHA_FIX}$",
                     fontsize=10)
        ax.legend(loc="lower right", handlelength=2.6, borderaxespad=0.6)
        fig.tight_layout()
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_ari_curve(path, "ari", "single linkage", filename="ari_path_single_linkage")

In [ ]:
_ = plot_ari_curve(path, "ari_average", "average linkage",
                   filename="ari_path_average_linkage")

In [ ]:
_ = plot_ari_curve(path, "ari_medoids", r"$K$-medoids",
                   filename="ari_path_kmedoids")

## The $(\alpha, K)$ grid

Section 5.3, Figures 5a, 5b and 10. The same grid as `assumptions.ipynb` — $\alpha \in \{0.1,
\dots, 10\}$, $K \in \{2, \dots, 10\}$, $R = 30$ repetitions per cell with new kernels drawn at
every repetition — so that the two sets of heatmaps can be read cell against cell.

It is run at the **last point of the path above**, $N = 800$ and $n = 1000$. That point is where
the path has climbed to $\mathrm{ARI} = 1$ on a mixture that is neither easy nor adversarial, so
it is a sample size known to be sufficient for at least one mixture, and the grid then says how
far that conclusion travels across $(\alpha, K)$. The price is that the horizon is no longer the
$n = 1500$ of `assumptions.ipynb`: the comparison with that notebook is at equal $(\alpha, K)$,
not at equal sequence length.

A single horizon is evaluated. A second one would not be free — an alignment is re-run at each
horizon, since nested prefixes share no work — and what it would show, the ARI rising with $n$,
is already read along the path above at ten lengths. The bottom rows ($\alpha \le 0.2$, where the
relaxation time of the kernels is largest) remain finite-$n$ statements.


In [ ]:
def sweep_ari(alphas, Ks, N, grid, R, seed=SEED + 2):
    """Mean ARI, exact-recovery rate and separation margin on the (alpha, K) grid.

    Returns a dict of (len(alphas), len(Ks), len(grid)) arrays. `ari` and `exact` describe the
    clustering, `margin` and `p_sep` the assumption, all four measured on the same data.
    """
    grid = np.atleast_1d(np.asarray(grid, dtype=np.int64))
    rng = np.random.default_rng(seed)
    shape = (len(alphas), len(Ks), grid.size)
    res = {key: np.empty(shape) for key in ("ari", "exact", "margin", "p_sep", "in", "out",
                                            "ari_average", "exact_average",
                                            "ari_medoids", "exact_medoids",
                                            "ari_k_hat", "k_hat", "p_k_hat")}
    for i, alpha in enumerate(tqdm(alphas, desc="alpha")):
        for j, K in enumerate(Ks):
            acc = {key: np.empty((R, grid.size)) for key in EVAL_KEYS}
            for r in range(R):
                mix = sample_mixture(K, N, int(grid[-1]), D_STATES, float(alpha), rng)
                out = evaluate_paths(mix["X"], mix["labels"], grid, K)
                for key in acc:
                    acc[key][r] = out[key]
            m = acc["out"] - acc["in"]
            res["ari"][i, j] = acc["ari"].mean(axis=0)
            res["exact"][i, j] = (acc["ari"] > 1 - EXACT_TOL).mean(axis=0)
            res["margin"][i, j] = np.median(m, axis=0)
            res["p_sep"][i, j] = np.mean(m > 0, axis=0)
            res["in"][i, j] = np.median(acc["in"], axis=0)
            res["out"][i, j] = np.median(acc["out"], axis=0)
            for est in ("average", "medoids"):
                res[f"ari_{est}"][i, j] = acc[f"ari_{est}"].mean(axis=0)
                res[f"exact_{est}"][i, j] = (acc[f"ari_{est}"] > 1 - EXACT_TOL).mean(axis=0)
            res["ari_k_hat"][i, j] = acc["ari_k_hat"].mean(axis=0)
            res["k_hat"][i, j] = np.median(acc["k_hat"], axis=0)
            res["p_k_hat"][i, j] = np.mean(acc["k_hat"] == K, axis=0)
    return res, grid

In [ ]:
t0 = time.time()
sweep, GRID_N = sweep_ari(ALPHAS, KS, N=N_SWEEP, grid=GRID_N, R=R_SWEEP)
print(f"sweep: {time.time() - t0:.0f}s")

for key, lab in (("ari", "mean ARI"), ("exact", "P(ARI = 1)"), ("p_sep", "P(margin > 0)")):
    Z = sweep[key][:, :, LAST]
    print(f"{lab:16s} in [{Z.min():.2f}, {Z.max():.2f}]")
print(f"{'median margin':16s} in [{sweep['margin'][:, :, LAST].min():+.2f}, "
      f"{sweep['margin'][:, :, LAST].max():+.2f}]")

In [ ]:
def _frame(ax, alphas, Ks, title):
    ax.set_xticks(range(len(Ks)), [str(K) for K in Ks])
    ax.set_yticks(range(len(alphas)), [f"{a:g}" for a in alphas])
    ax.set_xlabel(r"$K$")
    ax.set_ylabel(r"$\alpha$")
    ax.set_title(title, fontsize=9, pad=6)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color("0.7")


def _heat(fig, ax, Z, alphas, Ks, title, cbar_label, below=None, cmap=SEQUENTIAL_CMAP,
          vmin=0.0, vmax=1.0, fmt="{:.2f}", fmt_below="{:+.2f}"):
    """One heatmap of the (alpha, K) grid, optionally annotated with a second quantity below
    the first, in smaller type."""
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    _frame(ax, alphas, Ks, title)
    span = max(abs(vmin), abs(vmax))
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            light = (Z[i, j] - vmin) / (vmax - vmin) > 0.55 if cmap == SEQUENTIAL_CMAP \
                else abs(Z[i, j]) > 0.62 * span
            col = "white" if light else "0.25"
            dy = 0.13 if below is not None else 0.0
            ax.text(j, i + dy, fmt.format(Z[i, j]), ha="center", va="center",
                    fontsize=5.5, color=col)
            if below is not None:
                ax.text(j, i - 0.17, fmt_below.format(below[i, j]), ha="center", va="center",
                        fontsize=4.6, color=col, alpha=0.85)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label(cbar_label, fontsize=8)
    cb.outline.set_visible(False)
    return im


def plot_ari_grid(sweep, alphas, Ks, n, key, exact_key, name, g=LAST, filename=None):
    """One heatmap of the (alpha, K) grid, for one estimator.

    Colour and upper figure: the mean ARI, which measures how much of the partition is
    recovered. Lower figure, smaller: the probability of exact recovery, the event the
    consistency theorems bound. A cell can have a high mean ARI and never recover exactly --
    the union bound over the C(N, 2) pairs at work. The three estimators share the scale, so
    the three figures can be read cell against cell.

    The separation margin is not shown: it describes the assumption, not the performance, and
    `assumptions.ipynb` reports it on this very grid.
    """
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.4, 4.0))
        _heat(fig, ax, sweep[key][:, :, g], alphas, Ks,
              rf"{name}, $n = {n}$, $N = {N_SWEEP}$"
              "\n" rf"mean ARI, and $\widehat{{\mathbb{{P}}}}(\mathrm{{ARI}} = 1)$ below",
              "mean ARI", below=sweep[exact_key][:, :, g], fmt_below="{:.2f}")
        fig.tight_layout()
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_ari_grid(sweep, ALPHAS, KS, int(GRID_N[LAST]), "ari", "exact", "single linkage",
                  filename="ari_grid_single_linkage")

In [ ]:
_ = plot_ari_grid(sweep, ALPHAS, KS, int(GRID_N[LAST]), "ari_average", "exact_average",
                  "average linkage", filename="ari_grid_average_linkage")

In [ ]:
_ = plot_ari_grid(sweep, ALPHAS, KS, int(GRID_N[LAST]), "ari_medoids", "exact_medoids",
                  r"$K$-medoids", filename="ari_grid_kmedoids")

# what each estimator achieves, per K, averaged over alpha
print(f"{'K':>4}{'single':>10}{'average':>10}{'medoids':>10}   |{'P(=1) single':>14}{'P(=1) average':>15}{'P(=1) medoids':>15}")
for j, K in enumerate(KS):
    m = [sweep[k][:, j, LAST].mean() for k in ("ari", "ari_average", "ari_medoids")]
    e = [sweep[k][:, j, LAST].mean() for k in ("exact", "exact_average", "exact_medoids")]
    print(f"{K:>4}{m[0]:>10.2f}{m[1]:>10.2f}{m[2]:>10.2f}   |"
          f"{e[0]:>14.2f}{e[1]:>15.2f}{e[2]:>15.2f}")

## Estimating $K$: the largest-gap rule

Section 5.4, Figure 6. Everything above assumes $K$ known. The largest-gap estimator (6),
$\hat K = N - \arg\max_{1 \le \ell \le N-2}(h_{\ell+1} - h_\ell)$, reads it off the merge heights
themselves: the dendrogram is cut where it opens widest. It costs nothing here —
`single_linkage_tree` already returns the heights, so both partitions are scored on the same tree
and the same data as everything above.

Assumption 4, not Assumption 3, is what covers this rule, and `assumptions.ipynb` found it
essentially confined to $K \le 3$. The prediction is therefore that $\hat K$ collapses well
before the ARI at known $K$ does. Three quantities are reported: $\hat{\mathbb P}(\hat K = K)$,
the median $\hat K$ — which says *how* the rule fails, and not merely that it does — and the ARI
of the partition cut at $\hat K$, against the ARI at known $K$, whose difference is the price of
not knowing $K$.


In [ ]:
print(f"{'n':>7}{'ARI | K':>10}{'ARI | Khat':>12}"
      f"{'median Khat':>13}{'P(Khat = K)':>13}")
for t in range(path["n"].size):
    print(f"{path['n'][t]:>7}"
          f"{path['ari'][:, t].mean():>10.3f}{path['ari_k_hat'][:, t].mean():>12.3f}"
          f"{np.median(path['k_hat'][:, t]):>13.0f}"
          f"{np.mean(path['k_hat'][:, t] == K_FIX):>13.2f}")


In [ ]:
def plot_k_hat_grid(sweep, alphas, Ks, n, g=LAST, filename=None):
    """How often the largest-gap rule recovers K, over the (alpha, K) grid.

    Colour and upper figure: the probability of hitting K exactly. Lower figure: the median
    Khat, which says *how* the rule fails and not merely that it does. The ARI achieved at
    Khat is still measured -- it is in `sweep["ari_k_hat"]`, and the table below prints it
    against the ARI at known K -- but it is not drawn: the question here is whether K itself
    is recoverable.
    """
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.4, 4.0))
        _heat(fig, ax, sweep["p_k_hat"][:, :, g], alphas, Ks,
              rf"largest-gap rule, $n = {n}$, $N = {N_SWEEP}$"
              "\n" rf"$\widehat{{\mathbb{{P}}}}(\hat K = K)$, and median $\hat K$ below",
              "probability of recovering $K$", below=sweep["k_hat"][:, :, g],
              fmt_below="{:.0f}")
        fig.tight_layout()
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Recovery/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Recovery/{filename}.pdf")
        plt.show()
    return fig


_ = plot_k_hat_grid(sweep, ALPHAS, KS, int(GRID_N[LAST]), g=LAST, filename="k_hat_grid")

# the price of not knowing K, per K, averaged over alpha
print(f"{'K':>4}{'P(Khat = K)':>13}{'median Khat':>13}{'ARI | K':>10}{'ARI | Khat':>12}"
      f"{'loss':>8}")
for j, K in enumerate(KS):
    a_known = sweep["ari"][:, j, LAST].mean()
    a_hat = sweep["ari_k_hat"][:, j, LAST].mean()
    print(f"{K:>4}{sweep['p_k_hat'][:, j, LAST].mean():>13.2f}"
          f"{np.median(sweep['k_hat'][:, j, LAST]):>13.0f}"
          f"{a_known:>10.2f}{a_hat:>12.2f}{a_known - a_hat:>8.2f}")